# FIP 01 — Movement (motion energy) × RPE / value coding

Extends `fip_00_explore.ipynb`'s single-session FIP + motion-energy (ME) pipeline with the
analysis used for the FIP-only tonic-value / phasic-RPE figure (baseline AUC by consecutive
reward streak, outcome traces by RPE bin), applied to motion energy across all example FIP
channels. Two figures:

1. Session-averaged movement over time per RPE bin
2. Z-scored AUC per consecutive R-/R+ (`num_reward_past`)

Recipe traced directly from `rachel-analysis-utils`, `aind-dynamic-foraging-basic-analysis`,
`aind-dynamic-foraging-data-utils`, and (for the plotting conventions) the `DA_phasic_tonic`
repo — see the implementation plan (`fip_01` plan, 2026-09-14) for the full trace and the
deviations reviewed against that source. Setup (loading, curation, session select, trial
enrichment, example channels, motion energy) comes from the shared `fip_utils.py` module,
which `fip_00_explore.ipynb` and `fip_02_ne_only_events.ipynb` also use.

A third figure (NE-only onsets aligned to movement) originally lived here; it moved to
`fip_02_ne_only_events.ipynb`, which compares NE and DA transients directly rather than
treating all NE onsets alike.

## Imports & setup

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Shared fip_* setup: loading, curation, session select, trial enrichment, example channels,
# motion energy on the FIP clock. autoreload so editing fip_utils.py doesn't cost a kernel
# restart + a full session reload.
%load_ext autoreload
%autoreload 2
import fip_utils as fu

# FIP enrichment helpers (z-scoring, per-trial windowing, tonic/baseline removal) -- the same
# functions Rachel's own pipeline uses, not a reimplementation of them.
from aind_dynamic_foraging_data_utils import enrich_dfs

# Upstream PSTH-by-category plotter, reused for the RPE-binned movement panel (Figure 1).
from aind_dynamic_foraging_basic_analysis.plot import plot_fip as pf

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)


## Data loading and curation

`fu.load_curated_sessions` wraps `load_nwb_list` + `apply_curation_nwb_list`, including the
patch for `data_curation_helpers.apply_curation_nwb_list`'s missing private-helper imports (an
upstream bug) and the `drop_borderline=False` / free-the-raw-list memory fix.


In [ ]:
nwb_list = fu.load_curated_sessions()


## Data processing

### Select a session and build the signal table

Same example session as `fip_00_explore.ipynb` (`SESSION_IDX = 0`).
`require_trial_zero_gocue=True` additionally asserts `goCue_start_time_in_trial == 0` for every
trial, because `enrich_fip_in_df_trials`'s per-trial window math (below) assumes the go cue is
trial-local time zero.


In [ ]:
SESSION_IDX = 0
nwb, df_fip, df_trials = fu.select_session(nwb_list, SESSION_IDX, require_trial_zero_gocue=True)


In [ ]:
meta = fu.build_meta(df_fip)
print(f"{len(meta)} FIP series (pearsonR excluded)")
meta


### Trial enrichment — `num_reward_past` and `RPE-binned3`

`fu.enrich_trials` calls `rachel_analysis_utils.analysis_utils.enrich_df_trials` — the real
source of both columns — and falls back to a local reimplementation of just the two columns
this notebook needs if that module cannot be imported, so the notebook doesn't hard-depend on
the upstream import working.


In [ ]:
df_trials = fu.enrich_trials(df_trials)
nwb.df_trials = df_trials

RPE_binned3_label_names = df_trials["RPE-binned3"].cat.categories.astype(str).tolist()
print("RPE-binned3 categories:", RPE_binned3_label_names)
df_trials[["num_reward_past", "RPE-binned3"]].describe(include="all")


### Example signals

All three example FIP channels used in `fip_00_explore.ipynb` (NAc DA/dLight, PL/GCaMP,
NAc ACh/rAch), same `pick_example` selection logic.

In [ ]:
examples = fu.pick_examples(meta, df_fip)
pd.DataFrame(examples)[["label", "event", "channel", "region", "variant", "color"]]


### Motion energy on the FIP clock

`fu.locate_me_assets` finds this session's behavior-video and motion-energy assets;
`fu.motion_energy_to_session` puts per-frame ME on the first-go-cue-zeroed session clock,
including the leading-zero pad for aind-motion-energy's N−1 frame output.


In [ ]:
video_csv, me_path = fu.locate_me_assets(nwb.session_id)
print("video_csv:", video_csv)
print("me_path:  ", me_path)


In [ ]:
t_me, me, offset = fu.motion_energy_to_session(me_path, video_csv, df_trials)

# Verify before trusting: offset small & positive, ME/FIP time ranges overlap.
print(f"offset = {offset:.3f} s | ME {t_me.min():.1f}..{t_me.max():.1f}s | "
      f"FIP {df_fip['timestamps'].min():.1f}..{df_fip['timestamps'].max():.1f}s")


### Attach ME + run the real baseline/tonic-normalization pipeline

`fu.attach_me_to_df_fip` stores **raw** motion energy (not pre-z-scored) so ME flows through
`zscore_fip` / `enrich_fip_in_df_trials` / `remove_tonic_df_fip` exactly like a real FIP
channel -- these are Rachel's actual functions
(`aind_dynamic_foraging_data_utils.enrich_dfs`), channel-agnostic, run once over the whole
`df_fip` (every real channel + `"ME"` together). Produces `data_z` (per `(session, event)`
z-score) and, per trial, `data_z_{event}_baseline` (mean over the 1s immediately before that
trial's go cue) and `data_z_{event}_norm` (baseline-subtracted).


In [ ]:
df_fip = fu.attach_me_to_df_fip(df_fip, t_me, me, nwb.session_id)

# enrich_fip_in_df_trials z-scores internally (via zscore_fip) before windowing, so a separate
# upfront zscore_fip call isn't needed.
df_fip_z, df_trials_fip = enrich_dfs.enrich_fip_in_df_trials(df_fip, df_trials)
df_fip_tonic, df_trials, df_trials_fip = enrich_dfs.remove_tonic_df_fip(
    df_fip_z, df_trials, df_trials_fip)

nwb.df_fip = df_fip_z
nwb.df_trials = df_trials

# One pass produces baseline/norm columns for every channel in df_fip at once (channel-agnostic
# -- see title-cell plan note); check them all here, right after the pipeline runs, rather than
# failing deep inside a plotting loop later.
_baseline_cols = [f"data_z_{ex['event']}_baseline" for ex in examples] + ["data_z_ME_baseline"]
for _col in _baseline_cols:
    assert _col in df_trials.columns, f"missing {_col}"
print("baseline columns ready:", _baseline_cols)


## Figure 1 — session-averaged movement per RPE bin, all signals

One panel per signal (the 3 example FIP channels + motion energy). Same function, alignment
event, time window, censoring, and mako-by-ascending-bin color convention as `DA_phasic_tonic`'s
`PAC_2026.ipynb` recipe for the DA-channel version of this panel -- now looped over every
signal instead of just ME.

In [ ]:
rpe_dict = {
    label: df_trials.loc[df_trials["RPE-binned3"] == label, "choice_time_in_session"].dropna().to_numpy()
    for label in RPE_binned3_label_names
}
for label, times in rpe_dict.items():
    print(f"  RPE {label}: {len(times)} trials")

rpe_colors = dict(zip(RPE_binned3_label_names,
                      sns.color_palette("mako", len(RPE_binned3_label_names)).as_hex()))

plot_signals = examples + [{"label": "Motion energy", "event": "ME", "color": "#555555"}]

fig, axes = plt.subplots(1, len(plot_signals), figsize=(4 * len(plot_signals), 3.6), sharex=True)
for ax, sig in zip(np.atleast_1d(axes), plot_signals):
    pf.plot_fip_psth_compare_alignments(
        nwb, rpe_dict, channel=sig["event"], tw=[-1, 2], censor=True,
        data_column="data_z", extra_colors=rpe_colors, ax=ax, fig=fig)
    ax.set_title(sig["label"], loc="left", fontsize=10, color=sig["color"])
    ax.set_ylabel("signal (z)")
fig.suptitle(f"Signals aligned to choice, by RPE bin — {nwb.session_id}")
fig.tight_layout()
plt.show()

## Figure 2 — z-scored AUC per consecutive R-/R+, all signals

`num_reward_past_baseline = num_reward_past.shift(1)` pairs each trial's baseline value with
the reward-streak count *before* it was recorded (the `power_analysis.ipynb` /
`BWNM_explore_data.ipynb` convention -- mathematically the same pairing
`foraging_summary_plots.py::plot_baseline_corr` gets by shifting baseline the other way). Same
`sns.barplot(..., palette="vlag", hue=..., dodge=False)` recipe as that function's "Column 0",
applied to the real `remove_tonic_df_fip` baseline columns -- one panel per signal (the 3
example FIP channels + motion energy). `remove_tonic_df_fip` already ran over every channel in
`df_fip` in one pass (Attach ME section above), so every `data_z_{event}_baseline` column is
already there; no extra enrichment needed here.

In [ ]:
df_trials["num_reward_past_baseline"] = df_trials["num_reward_past"].shift(1)
df_bl = df_trials.query("num_reward_past_baseline > -7 and num_reward_past_baseline < 7")

baseline_signals = (
    [(ex["label"], f"data_z_{ex['event']}_baseline", ex["color"]) for ex in examples]
    + [("Motion energy", "data_z_ME_baseline", "#555555")]
)

fig, axes = plt.subplots(1, len(baseline_signals), figsize=(4 * len(baseline_signals), 4))
for ax, (label, col, color) in zip(np.atleast_1d(axes), baseline_signals):
    assert col in df_trials.columns, f"missing {col}"
    sns.barplot(x="num_reward_past_baseline", y=col, data=df_bl,
               palette="vlag", hue="num_reward_past_baseline", dodge=False, ax=ax)
    ax.legend().remove()
    ax.set_xlabel("# consecutive R-/R+ trials\n(num_reward_past, shifted)")
    ax.set_ylabel("baseline (z)")
    ax.set_title(label, color=color)
fig.suptitle(f"Baseline AUC vs. consecutive reward streak — {nwb.session_id}")
fig.tight_layout()
plt.show()